# 14 - Codility 3: primes, sieve, Euclid, Fibonacci

**Source:** *Codility Lessons* chapters 10-13. Practice: `count_factors`, `min_perimeter_rectangle`, `chocolates_by_numbers`,
`count_semiprimes`, `ladder` in `codility/practice_04_numbers_search_greedy_dp.py`.

**Bugs / traps found in the PDF code (fixed below):**
* Euclid by subtraction (12.1): the recursive calls lack `return` -> the function returns `None`.
* Factorisation (11.3): `x /= F[x]` is float division in Python 3 -> use `x //= F[x]`.
* `sieve(n)`: for `n < 2` the line `sieve[1] = False` raises `IndexError`.
* Python has **unbounded integers** (no overflow), but huge numbers are slower, so Codility's "return modulo 2^k"
  tasks should still reduce *inside* the loop.

## 1. Divisors, primes, composites
A divisor `a` of `n` pairs with `n / a`; one of the two is <= sqrt(n). So scanning `1..sqrt(n)` finds all divisors: **O(sqrt n)**.
Use `math.isqrt(n)` (exact integer square root) instead of `int(n ** 0.5)` (float rounding for big n).

In [1]:
import math
import random
rng = random.Random(3)

def divisors_count(n):
    i, result = 1, 0
    while i * i < n:
        if n % i == 0:
            result += 2
        i += 1
    if i * i == n:                       # perfect square: the symmetric divisor is itself, count once
        result += 1
    return result

def divisors_list(n):
    small = [i for i in range(1, math.isqrt(n) + 1) if n % i == 0]
    return sorted(set(small + [n // d for d in small]))

def is_prime(n):
    if n < 2:
        return False                     # the PDF assumes n >= 2
    i = 2
    while i * i <= n:
        if n % i == 0:
            return False
        i += 1
    return True

assert divisors_count(36) == 9 and divisors_list(36) == [1, 2, 3, 4, 6, 9, 12, 18, 36]
assert [p for p in range(30) if is_prime(p)] == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
for n in range(1, 500):
    assert divisors_count(n) == len(divisors_list(n))
print(divisors_count(2_147_483_647), "<- prime near 2^31, found in ~46 000 steps")

2 <- prime near 2^31, found in ~46 000 steps


### Reversing coins (PDF 10.3)
`n` coins, person `i` flips every multiple of `i`. Coin `k` is flipped once per divisor of `k`, so it shows tails iff it
has an **odd** number of divisors, i.e. iff `k` is a **perfect square**. Answer: `isqrt(n)` - O(1) instead of O(n log n).

In [2]:
def coins_simulation(n):
    coin = [0] * (n + 1)
    for i in range(1, n + 1):
        for k in range(i, n + 1, i):
            coin[k] ^= 1
    return sum(coin)

def coins_formula(n):
    return math.isqrt(n)

assert all(coins_simulation(n) == coins_formula(n) for n in range(1, 200))

**Min perimeter rectangle** (area N): sides `a <= sqrt(N)` and `N / a`; scan `a` downwards from `isqrt(N)` to the first
divisor. **Peaks/flags**-type tasks use the same "divisors <= sqrt" trick.

## 2. Sieve of Eratosthenes
Cross out multiples of each prime `p <= sqrt(n)`, starting at `p*p` (smaller multiples were already removed by smaller
primes). **O(n log log n)** time, O(n) memory.

In [3]:
def sieve(n):
    flags = [True] * (n + 1)
    flags[:2] = [False] * min(2, n + 1)               # 0 and 1 are not prime (safe for n < 2, unlike the PDF)
    i = 2
    while i * i <= n:
        if flags[i]:
            for k in range(i * i, n + 1, i):
                flags[k] = False
        i += 1
    return flags

def sieve_fast(n):
    """Slice assignment on a bytearray: ~50x faster in CPython."""
    flags = bytearray([1]) * (n + 1)
    flags[:2] = b"\x00" * min(2, n + 1)
    for i in range(2, math.isqrt(n) + 1):
        if flags[i]:
            flags[i * i :: i] = bytes(len(range(i * i, n + 1, i)))
    return flags

import timeit
assert [i for i, f in enumerate(sieve(50)) if f] == [i for i in range(51) if is_prime(i)]
assert list(map(bool, sieve_fast(10_000))) == sieve(10_000)
assert sieve(0) == [False] and sieve(1) == [False, False]
N = 2_000_000
print(f"sieve      {timeit.timeit(lambda: sieve(N), number=1):.3f}s")
print(f"sieve_fast {timeit.timeit(lambda: sieve_fast(N), number=1):.3f}s   primes below {N}: {sum(sieve_fast(N))}")

sieve      0.181s


sieve_fast 0.009s   primes below 2000000: 148933


### Factorisation with the "smallest prime factor" array
Modify the sieve: `F[k]` = the smallest prime dividing `k` (0 for primes). Then factorise `x` in **O(log x)** by
repeated division. The same array also answers "is `x` a semiprime?" (two prime factors) - the heart of *CountSemiprimes*.

In [4]:
def array_f(n):
    F = [0] * (n + 1)
    i = 2
    while i * i <= n:
        if F[i] == 0:
            for k in range(i * i, n + 1, i):
                if F[k] == 0:
                    F[k] = i
        i += 1
    return F

def factorization(x, F):
    prime_factors = []
    while F[x] > 0:
        prime_factors.append(F[x])
        x //= F[x]                       # the PDF's `x /= F[x]` produces floats in Python 3
    prime_factors.append(x)
    return prime_factors

F = array_f(100)
assert F[20] == 2 and F[15] == 3 and F[13] == 0
assert factorization(84, F) == [2, 2, 3, 7] and factorization(97, F) == [97]
assert math.prod(factorization(96, F)) == 96

In [5]:
# From factorisation to counting semiprimes in ranges (prefix sums on a 0/1 array):
def semiprime_flags(n):
    F = array_f(n)
    flags = [0] * (n + 1)
    for x in range(4, n + 1):
        if F[x] > 0 and F[x // F[x]] == 0:          # composite whose cofactor is prime (F == 0)
            flags[x] = 1
    return flags

flags = semiprime_flags(26)
print([i for i, f in enumerate(flags) if f])        # 4 6 9 10 14 15 21 22 25 26

[4, 6, 9, 10, 14, 15, 21, 22, 25, 26]


## 3. Euclidean algorithm
`gcd(a, b)`: by subtraction O(n) (!), by division **O(log(a + b))** (worst case consecutive Fibonacci numbers),
binary Euclid (shifts and subtraction; good for huge integers). `lcm(a, b) = a * b // gcd(a, b)` - divide *first*
(`a // gcd * b`) in fixed-width languages to avoid overflow. Python: `math.gcd`, `math.lcm` (3.9+) accept many arguments.

In [6]:
def gcd_subtraction(a, b):
    if a == b:
        return a
    if a > b:
        return gcd_subtraction(a - b, b)          # `return` was missing in the PDF!
    return gcd_subtraction(a, b - a)

def gcd_division(a, b):
    while b:
        a, b = b, a % b
    return a

def gcd_binary(a, b, res=1):
    if a == b:
        return res * a
    if a % 2 == 0 and b % 2 == 0:
        return gcd_binary(a // 2, b // 2, 2 * res)
    if a % 2 == 0:
        return gcd_binary(a // 2, b, res)
    if b % 2 == 0:
        return gcd_binary(a, b // 2, res)
    if a > b:
        return gcd_binary(a - b, b, res)
    return gcd_binary(a, b - a, res)

for _ in range(300):
    a, b = rng.randint(1, 500), rng.randint(1, 500)
    assert gcd_subtraction(a, b) == gcd_division(a, b) == gcd_binary(a, b) == math.gcd(a, b)
assert math.lcm(4, 6, 10) == 60 and math.gcd(24, 9) == 3

def lcm_many(*nums):
    result = 1
    for x in nums:
        result = result // math.gcd(result, x) * x
    return result

assert lcm_many(4, 6, 10) == 60

**Chocolates by numbers:** eating every `M`-th chocolate on a circle of `N` visits `N / gcd(N, M)` distinct positions.

In [7]:
def chocolates_simulation(N, M):
    eaten, pos = set(), 0
    while pos not in eaten:
        eaten.add(pos)
        pos = (pos + M) % N
    return len(eaten)

assert all(chocolates_simulation(N, M) == N // math.gcd(N, M) for N in range(1, 40) for M in range(1, 60))
print(chocolates_simulation(10, 4))

5


Other Euclid-related facts: `gcd(a, b) = gcd(b, a mod b)`; Bezout: `a*x + b*y = gcd(a, b)` (extended Euclid, gives
modular inverses); `gcd` of a list = fold; two numbers have the same prime divisors iff after dividing both by
their gcd repeatedly you reach 1 (*CommonPrimeDivisors*).

## 4. Fibonacci numbers
`F0 = 0, F1 = 1, Fn = Fn-1 + Fn-2`. Naive recursion is **exponential** (O(phi^n)); a loop is O(n); fast doubling or
matrix power are O(log n); Binet's formula is O(1) but inexact for large n (floats).

In [8]:
from functools import lru_cache

def fib_recursive(n):
    return n if n <= 1 else fib_recursive(n - 1) + fib_recursive(n - 2)

@lru_cache(maxsize=None)
def fib_memo(n):                         # still recursion-limited for n > ~900
    return n if n <= 1 else fib_memo(n - 1) + fib_memo(n - 2)

def fib_dynamic(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

def fib_doubling(n):
    """Fast doubling: F(2k) = F(k) * (2F(k+1) - F(k)); F(2k+1) = F(k)^2 + F(k+1)^2."""
    def fd(k):
        if k == 0:
            return 0, 1
        a, b = fd(k // 2)
        c = a * (2 * b - a)
        d = a * a + b * b
        return (d, c + d) if k % 2 else (c, d)
    return fd(n)[0]

def fib_binet(n):
    phi = (1 + 5 ** 0.5) / 2
    return round(phi ** n / 5 ** 0.5)

assert [fib_dynamic(i) for i in range(12)] == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]
assert all(fib_recursive(i) == fib_memo(i) == fib_dynamic(i) == fib_doubling(i) for i in range(20))
print("Binet exact up to n =", next(n for n in range(100) if fib_binet(n) != fib_dynamic(n)) - 1)
print(f"recursive fib(27): {timeit.timeit(lambda: fib_recursive(27), number=1):.3f}s | "
      f"loop fib(100000): {timeit.timeit(lambda: fib_dynamic(100_000), number=1):.3f}s | "
      f"doubling fib(100000): {timeit.timeit(lambda: fib_doubling(100_000), number=1):.5f}s")

Binet exact up to n = 70
recursive fib(27): 0.019s | loop fib(100000): 0.081s | doubling fib(100000): 0.00114s


### Sum of two Fibonacci numbers (PDF 13.2)
Fewer than 31 Fibonacci numbers are <= 10^6, so mark all pairwise sums in an array of size `m + 1`, then each query is O(1).

In [9]:
def two_fib_sums(xs, m):
    fibs = [0, 1]
    while fibs[-1] <= m:
        fibs.append(fibs[-1] + fibs[-2])
    reachable = [False] * (m + 1)
    for i, a in enumerate(fibs):
        for b in fibs[i:]:
            if a + b <= m:
                reachable[a + b] = True
    return [reachable[x] for x in xs]

print(two_fib_sums([1, 4, 7, 17, 100, 1000], 1000))        # 17 and 100 are not sums of two Fibonacci numbers

# brute-force check of the whole range
fib_set = {fib_dynamic(i) for i in range(30)}
brute = [any(x - f in fib_set for f in fib_set if f <= x) for x in range(1, 1001)]
assert two_fib_sums(list(range(1, 1001)), 1000) == brute
assert two_fib_sums([4, 7, 17, 100, 1000], 1000) == [True, True, False, False, True]

[True, True, True, False, False, True]
